# Deep Dive: Evaluator-Optimizer

## 📋 Summary

The **evaluator-optimizer** pattern is a **generate → evaluate → improve** loop with two roles:

- the **generator** (optimizer) writes a candidate,
- the **evaluator** checks it against explicit criteria and explains what is wrong,
- the evaluator's feedback goes **back to the generator**, which tries again,
- the loop stops when the candidate **passes** or after **`MAX_RETRIES`** attempts.

```
        ┌──────────── feedback ────────────┐
        ▼                                   │
   GENERATOR ──▶ candidate ──▶ EVALUATOR ──┴─▶ pass? ──yes──▶ done
                                                  │
                                                  └─no, and retries left ──▶ loop
```

It works when **"good" can be described** (a list of constraints) and it is **easier to check an answer than to write it right the first time**.
The loop is *bounded*: without `MAX_RETRIES` an impossible request would loop (and bill you) forever.

| Design choice | Options |
|---|---|
| Who evaluates? | **Code** for hard, countable rules (length, banned words). **LLM judge** for soft rules (tone, quality score). Best: both. |
| What does feedback look like? | A **list of concrete failures**, not "try harder" |
| When to stop? | All checks pass · max retries · no progress |
| What to return? | Final answer **and** whether it passed, how many iterations it took, and the history |

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | The generator (and how it uses feedback) | 🟢 Beginner |
| 2 | The evaluator as an LLM judge with a structured verdict | 🟡 Intermediate |
| 3 | Hard checks in code + soft checks by LLM | 🟡 Intermediate |
| 4 | The bounded loop with history and best-so-far | 🔴 Advanced |
| 5 | Failure modes: conflicting constraints, oscillation | 🔴 Advanced |
| 6 | Exercises | 🏋️ Practice |

**How to use this notebook:** run the cells in order. Cells marked `# TODO` are for you to fill in.
The notebook calls the OpenAI API with a cheap model; loops multiply calls, so `MAX_RETRIES` also protects your wallet.

## 0. Setup

In [ ]:
# Key input: in Colab, add OPENAI_API_KEY in the 🔑 Secrets panel (left sidebar).
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [ ]:
import json

from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4.1-nano"  # cheap model, plenty for these experiments


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def call_llm_json(system_prompt: str, user_prompt: str) -> dict:
    """Forces a JSON object back (the prompt must mention JSON)."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)


def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))

## 1. The Generator 🟢

**Running example:** write a **tagline for a reusable water bottle** brand called *Drift*.

The generator has two modes:
- **first attempt**: only the brief,
- **retry**: the brief **plus its previous attempt and the list of failed checks**, with the instruction to fix *all* of them.

Giving the generator its *own previous attempt* matters: the fix is an edit, not a fresh guess.

In [ ]:
BRAND = "Drift"
BRIEF = f"Write a tagline for {BRAND}, a reusable water bottle brand for people who commute by bike."


def generate_tagline(brief: str, previous: str | None = None, feedback: str | None = None) -> str:
    system = "You are a copywriter. Reply with ONE tagline only: no quotes, no explanation."
    user = f"Brief: {brief}"
    if feedback:
        user += (f"\n\nYour previous attempt: {previous}\n"
                 f"It FAILED these checks:\n{feedback}\n"
                 "Write a new tagline that fixes ALL of them.")
    return call_llm(system, user, temperature=0.8).strip('"')


print(generate_tagline(BRIEF))

## 2. The Evaluator as an LLM Judge 🟡

A useful judge does not answer "good" or "bad". It answers **per criterion**, with a **reason**, and gives a **score**. That structure becomes
the generator's feedback. We force it with JSON so code can read the verdict.

In [ ]:
SOFT_CRITERIA = [
    "Playful, friendly tone",
    "Suggests reusability or sustainability",
    "Would make a stranger smile",
]
MIN_SCORE = 7   # like "taste must be rated 7/10 or higher"


def judge_soft(tagline: str) -> dict:
    system = f"""You are a strict brand judge. Evaluate the tagline against EACH criterion, then give an overall quality score from 1 to 10.
Criteria: {SOFT_CRITERIA}
Return JSON exactly like:
{{"criteria": [{{"criterion": "...", "pass": true, "reason": "one short sentence"}}], "score": 7}}"""
    return call_llm_json(system, f"Tagline: {tagline}")


verdict = judge_soft("Drift: sip, ride, refill, repeat.")
print(json.dumps(verdict, indent=2))

### 🧐 Observation
- The same tagline can get different scores on different runs. What does that say about using a *score* as a stopping condition, and what did we do to reduce the variance (hint: temperature)?
- Why is a **reason per criterion** more useful to the generator than only a score of 5/10?

## 3. Hard Checks in Code + Soft Checks by LLM 🟡

LLMs are **poor at counting** ("at most 8 words") and can be lenient about rules like "must include the brand". **Anything that can be checked
with code should be checked with code**: it is free, instant and exact. Keep the LLM judge for what needs judgment.

Two more advantages:
- **Cheap checks first:** if a hard rule fails, skip the LLM judge (saves a call) and return the failure immediately.
- The evaluator returns one uniform result: `{"passed", "failures", "score"}`.

In [ ]:
DEFAULT_RULES = {"max_words": 8, "banned": ["best", "revolutionary", "cheap"]}


def check_hard(tagline: str, max_words: int, banned: list) -> list:
    failures = []
    words = tagline.split()
    if len(words) > max_words:
        failures.append(f"Too long: {len(words)} words (maximum {max_words}).")
    if BRAND.lower() not in tagline.lower():
        failures.append(f"Must contain the brand name '{BRAND}'.")
    for w in banned:
        if w.lower() in tagline.lower():
            failures.append(f"Contains the banned word '{w}'.")
    return failures


def evaluate(tagline: str, **rules) -> dict:
    rules = {**DEFAULT_RULES, **rules}
    failures = check_hard(tagline, rules["max_words"], rules["banned"])
    if failures:                                   # cheap checks first
        return {"passed": False, "failures": failures, "score": None}

    verdict = judge_soft(tagline)
    failures = [f"{c['criterion']}: {c['reason']}" for c in verdict["criteria"] if not c["pass"]]
    if verdict["score"] < MIN_SCORE:
        failures.append(f"Overall score {verdict['score']}/10 is below the minimum of {MIN_SCORE}.")
    return {"passed": not failures, "failures": failures, "score": verdict["score"]}


for t in ["The best bottle ever made for cyclists everywhere", "Drift: sip, ride, refill, repeat."]:
    print(t, "→", evaluate(t))

## 4. The Bounded Loop with History and Best-so-far 🔴

Now we connect generator and evaluator. Details that make the loop trustworthy:

- `max_retries` **bounds** the loop.
- We keep a **history** of every attempt (candidate, failures, score) for debugging.
- We track the **best-so-far** (fewest failures) so that when retries run out we return the closest attempt, not the last one.
- We return a **dict** saying whether it *passed*, not only the text: the caller must know if the output is trustworthy.

In [ ]:
MAX_RETRIES = 4


def optimize(brief: str, max_retries: int = MAX_RETRIES, verbose: bool = True, **rules) -> dict:
    history, best = [], None
    previous = feedback = None

    for attempt in range(1, max_retries + 1):
        candidate = generate_tagline(brief, previous, feedback)
        result = evaluate(candidate, **rules)
        record = {"attempt": attempt, "candidate": candidate, **result}
        history.append(record)

        if verbose:
            status = "✅ PASS" if result["passed"] else f"❌ {len(result['failures'])} failure(s)"
            print(f"Attempt {attempt}: {candidate!r} → {status}")
            for f in result["failures"]:
                print(f"     - {f}")

        if result["passed"]:
            return {"final": candidate, "passed": True, "iterations": attempt, "history": history}

        if best is None or len(result["failures"]) < len(best["failures"]):
            best = record
        previous = candidate
        feedback = "\n".join(f"- {f}" for f in result["failures"])

    return {"final": best["candidate"], "passed": False, "iterations": max_retries, "history": history}


run = optimize(BRIEF)
print("\nFinal:", run["final"], "| passed:", run["passed"], "| iterations:", run["iterations"])

### 🧐 Observation
- Did the loop need more than one attempt? What did the **feedback** change between attempts?
- Why return `passed: False` together with the *best* attempt instead of raising an error when retries run out? Who decides what happens next?

## 5. Failure Modes: Conflicting Constraints and Oscillation 🔴

A feedback loop is only as good as its **specification**. Free demo of the classic failure: **two constraints that contradict each other**
(the brand name is required, but also banned). No number of retries can fix that.

In [ ]:
impossible = optimize(BRIEF, max_retries=3, banned=["drift"])
print("\nPassed:", impossible["passed"], "| returned best attempt:", impossible["final"])

| Failure mode | What you see | Fix |
|---|---|---|
| **Conflicting / impossible constraints** | Every attempt fails the same rules until `MAX_RETRIES` | Validate the constraint set once up front; on failure **report which constraints cannot be met together** |
| **Oscillation** | Fixing rule A breaks rule B, then the reverse | Feed back the **whole history** of failures, and keep the constraints that already pass in the prompt |
| **Lenient judge** | Everything passes at 8-9/10 | Strict judge prompt, temperature 0, hard checks in code, calibrate on examples that *should* fail |
| **Harsh judge** | Nothing ever passes | Lower the bar, split criteria into must-have and nice-to-have |
| **Generator ignores feedback** | Same failure across attempts | Include the previous attempt, make each failure specific and actionable |
| **Cost blow-up** | Many retries × two calls each | Cheap checks first, cap retries, use a smaller model for the judge |

### 🧐 Observation
- Which of these failure modes would be **invisible** if the function returned only the final string? That is why we return `passed` and `history`.
- How would you detect "no progress" (the same failures for 2 attempts in a row) and stop early?

## 6. Exercises 🏋️

### Exercise 1: A domain-independent loop
`optimize` only knows about taglines. Write **`evaluator_optimizer(generate_fn, evaluate_fn, max_retries)`** so it works for **any** generator and evaluator:
- `generate_fn(previous, feedback)` returns a candidate,
- `evaluate_fn(candidate)` returns `{"passed", "failures", ...}`,
- it returns the same dict as `optimize` (`final`, `passed`, `iterations`, `history`),
- **stop early** if two consecutive attempts have exactly the same failures ("no progress").

In [ ]:
def evaluator_optimizer(generate_fn, evaluate_fn, max_retries: int = 4) -> dict:
    history = []
    previous = feedback = None
    # TODO: loop up to max_retries, evaluate each candidate, keep the best-so-far,
    #       stop when it passes OR when there is no progress
    return {"final": None, "passed": False, "iterations": 0, "history": history}


# TODO: wire it to the tagline generator and evaluator and check it reproduces `optimize`

### Exercise 2: Plug in a new domain
**Domain:** the **subject line of a newsletter email**.
Hard rules (code): at most 50 characters, no ALL-CAPS word longer than 3 letters, at most one exclamation mark.
Soft rules (LLM judge): creates curiosity without being clickbait, matches a friendly-professional tone; overall score of at least 7.

Reuse your generic loop from Exercise 1.

In [ ]:
NEWSLETTER_BRIEF = "Subject line for the monthly newsletter of a local library announcing new audiobooks."


def generate_subject(previous, feedback) -> str:
    # TODO
    pass


def evaluate_subject(subject: str) -> dict:
    # TODO: hard checks in code first, then the LLM judge; return {"passed", "failures", "score"}
    pass


# TODO: result = evaluator_optimizer(generate_subject, evaluate_subject)

### Exercise 3: A multi-constraint stretch challenge
The Udacity class file `udacity_files/class_06-evaluator optimizer.py` sets up a request with **many constraints** and `MAX_RETRIES = 5`.
Apply the pattern to it yourself. Think first about which constraints are **countable** (code can check them, or a nutrition estimate can be requested as JSON)
and which are **subjective** (the taste rating).

In [ ]:
MAX_RETRIES = 5

RECIPE_REQUEST = {
    "base_dish": "pasta",
    "constraints": [
        "gluten-free",
        "vegan",
        "under 500 calories per serving",
        "high protein (>15g per serving)",
        "no coconut",
        "taste must be rated 7/10 or higher",
    ],
}

# TODO: 1. Write the generator (it should receive the previous recipe + the failed constraints)
# TODO: 2. Write the evaluator: one JSON verdict per constraint, with a reason
# TODO: 3. Decide which checks you can do in code (e.g. banned ingredient words) and which need the LLM
# TODO: 4. Run the loop and print the history: which constraint was hardest?

## 🏁 Summary & Key Takeaways

- **Evaluator-optimizer** = generate → evaluate → feedback → regenerate, **bounded** by `MAX_RETRIES`.
- Good feedback is **specific and per-criterion** ("Too long: 11 words, max 8"), not "try again".
- Check with **code** what code can check; use an **LLM judge** for subjective criteria, and run the cheap checks **first**.
- Give the generator its **previous attempt** so it edits instead of guessing anew.
- Always return **whether it passed**, the **iterations** and the **history**. Keep the **best-so-far** for when retries run out.
- A loop cannot fix a **bad spec**: conflicting constraints, a lenient or harsh judge and oscillation are design problems.

### What's next?
- **Deep Dive 6: Orchestrator-Workers**: when you do not even know the steps in advance, let an orchestrator plan the subtasks and delegate them to workers.